# 🏥 Supervisor Agent Control Tower — Thorough Diagnostic Notebook

**Purpose:** End-to-end health check of every module, layer, and integration point.  
Run all cells top-to-bottom. Each section prints ✅ / ❌ / ⚠️ and a final summary table at the end.

| Section | What is tested |
|---------|---------------|
| 0 | Python environment, PYTHONPATH, installed packages |
| 1 | Config module — all settings fields |
| 2 | Pydantic models — enums, validators |
| 3 | Excel store — CRUD + all 5 analytics methods |
| 4 | Orchestrator — routing all 4 agent types |
| 5 | Rules engine — all 4 tools against seed records |
| 6 | LLM client — mock mode + backend selection |
| 7 | Judge module — mock evaluation |
| 8 | Synthesizer + Scorecard |
| 9 | InsightsEngine — drift, readiness, recommendations |
| 10 | End-to-end pipeline — all 12 seed records |
| 11 | Final summary report |

In [ ]:
"""
SETUP — run this cell first every time.
Sets PYTHONPATH, shared results tracker, and helper functions.
"""
import sys, os, pathlib

# ── Locate project root and src ──────────────────────────────────────────────
HERE = pathlib.Path().resolve()
# Walk up until we find app.py (project root)
root = HERE
for _ in range(5):
    if (root / "app.py").exists():
        break
    root = root.parent

SRC = str(root / "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)

# Also set env var so sub-processes pick it up
os.environ["PYTHONPATH"] = SRC

# Change working dir so relative paths (data/, .env) resolve correctly
os.chdir(root)

# ── Shared results collector ─────────────────────────────────────────────────
RESULTS: list[dict] = []   # {"section": str, "check": str, "status": str, "detail": str}

def ok(section: str, check: str, detail: str = ""):
    RESULTS.append({"section": section, "check": check, "status": "✅ PASS", "detail": detail})
    print(f"  ✅ {check}: {detail}")

def fail(section: str, check: str, detail: str = ""):
    RESULTS.append({"section": section, "check": check, "status": "❌ FAIL", "detail": detail})
    print(f"  ❌ {check}: {detail}")

def warn(section: str, check: str, detail: str = ""):
    RESULTS.append({"section": section, "check": check, "status": "⚠️ WARN", "detail": detail})
    print(f"  ⚠️ {check}: {detail}")

print(f"✔ Project root  : {root}")
print(f"✔ src on path   : {SRC}")
print(f"✔ cwd           : {os.getcwd()}")
print(f"✔ Python        : {sys.version}")

## 📦 Section 0 — Environment & Installed Packages

In [ ]:
import platform, importlib.metadata as im

S = "Environment"
print(f"=== Section 0: Environment & Packages ===\n")

# Python version
py_ver = sys.version_info
if py_ver >= (3, 11):
    ok(S, "Python version", f"{sys.version.split()[0]}")
else:
    warn(S, "Python version", f"{sys.version.split()[0]} — recommend 3.11+")

# OS
ok(S, "OS", platform.platform())

# Virtual-env detection
venv = os.environ.get("VIRTUAL_ENV") or os.environ.get("CONDA_DEFAULT_ENV") or "not detected"
ok(S, "Virtual env", venv)

# PYTHONPATH
ok(S, "PYTHONPATH", SRC)

# Required packages
REQUIRED = [
    "streamlit", "pydantic", "pydantic_settings",
    "openai", "httpx", "requests",
    "openpyxl", "plotly",
    "sqlalchemy",
]
missing = []
for pkg in REQUIRED:
    try:
        ver = im.version(pkg)
        ok(S, f"pkg:{pkg}", ver)
    except im.PackageNotFoundError:
        fail(S, f"pkg:{pkg}", "NOT INSTALLED")
        missing.append(pkg)

if missing:
    print(f"\n⚠️  Missing packages: {missing}")
    print(f"   Install with: pip install {' '.join(missing)}")
else:
    print("\n✅ All required packages present.")

## ⚙️ Section 1 — Config Module

In [ ]:
from supervisor_control_tower.config import Settings

S = "Config"
print("=== Section 1: Config Module ===\n")

try:
    settings = Settings()
    ok(S, "Settings instantiated", "")
except Exception as e:
    fail(S, "Settings instantiated", str(e))
    raise

# Storage
ok(S, "storage_backend", settings.storage_backend)
ok(S, "excel_store_path", settings.excel_store_path)

# Excel file exists?
excel_path = pathlib.Path(settings.excel_store_path)
if excel_path.exists():
    ok(S, "excel file on disk", str(excel_path.resolve()))
else:
    warn(S, "excel file on disk", f"not found at {excel_path} — run seed_data.py first")

# LLM config
ok(S, "mock_llm", str(settings.mock_llm))
ok(S, "azure_llm_enabled", str(settings.azure_llm_enabled))
ok(S, "llm_model", settings.llm_model)

if settings.azure_llm_enabled:
    ok(S, "azure_endpoint", settings.azure_endpoint[:40] + "...")
    ok(S, "azure_llm_model", settings.azure_llm_model)
    ok(S, "azure_api_version", settings.azure_api_version)
    if settings.azure_client_id:
        ok(S, "azure_client_id", settings.azure_client_id[:8] + "***")
    else:
        fail(S, "azure_client_id", "empty")
    if settings.azure_client_secret:
        ok(S, "azure_client_secret", "***set***")
    else:
        fail(S, "azure_client_secret", "empty")
elif settings.openai_api_key:
    ok(S, "openai_api_key", "***set***")
else:
    if settings.mock_llm:
        ok(S, "LLM backend", "mock mode — no real key needed")
    else:
        warn(S, "LLM backend", "MOCK_LLM=false but no Azure or OpenAI key configured")

# Thresholds
ok(S, "high_confidence_threshold", str(settings.high_confidence_threshold))
ok(S, "minimum_confidence_threshold", str(settings.minimum_confidence_threshold))

# Auth
ok(S, "auth_enabled", str(settings.auth_enabled))
ok(S, "demo_auth", str(settings.demo_auth))

print("\n✅ Config section complete.")

## 📐 Section 2 — Pydantic Models

In [ ]:
from supervisor_control_tower.models import (
    AgentCode, ToolCode, Verdict, Severity, ExecutionStatus,
    NormalizedRecord, RoutingDecision, RuleResultModel,
    LlmJudgementResult, JudgeFinding, FinalSynthesis,
    ValidationRunResult, AppUser, TOOL_TO_AGENT, AGENT_TO_TOOL,
)
from pydantic import ValidationError

S = "Models"
print("=== Section 2: Pydantic Models ===\n")

# Enums
ok(S, "AgentCode values", str(list(AgentCode)))
ok(S, "ToolCode values", str(list(ToolCode)))
ok(S, "Verdict values", str(list(Verdict)))
ok(S, "Severity values", str(list(Severity)))

# TOOL_TO_AGENT mapping
assert len(TOOL_TO_AGENT) == 4, "Expected 4 tool→agent mappings"
assert len(AGENT_TO_TOOL) == 4, "Expected 4 agent→tool mappings"
ok(S, "TOOL_TO_AGENT size", "4 entries")
ok(S, "AGENT_TO_TOOL size", "4 entries")

# NormalizedRecord
try:
    rec = NormalizedRecord(
        record_id="test-001",
        external_reference="EXT-001",
        source_system="github_actions",
        record_type="pipeline_failure",
        record_title="Test record",
        payload={"pipeline_run_id": "run-1"},
        metadata={"environment": "prod"},
    )
    ok(S, "NormalizedRecord", f"id={rec.record_id}, src={rec.source_system}")
except ValidationError as e:
    fail(S, "NormalizedRecord", str(e))

# RoutingDecision
try:
    rd = RoutingDecision(
        selected_tool=ToolCode.PIPELINE,
        detected_agent_code=AgentCode.PIPELINE_TROUBLESHOOTING,
        reason="CI/CD source",
        confidence=0.96,
    )
    ok(S, "RoutingDecision", f"tool={rd.selected_tool}, conf={rd.confidence}")
except ValidationError as e:
    fail(S, "RoutingDecision", str(e))

# RuleResultModel
try:
    rr = RuleResultModel(
        rule_code="PIPE-001", rule_name="Pipeline ID present",
        severity=Severity.CRITICAL, passed=True,
        message="Pipeline run ID found", tag="PIPELINE_ID", evidence={"found": True},
    )
    ok(S, "RuleResultModel", f"code={rr.rule_code}, passed={rr.passed}")
except ValidationError as e:
    fail(S, "RuleResultModel", str(e))

# LlmJudgementResult
try:
    jf = JudgeFinding(severity=Severity.MEDIUM, message="Missing log evidence")
    jr = LlmJudgementResult(
        verdict=Verdict.WARNING, confidence=0.72,
        reason="Some issues found", findings=[jf], focus_area_addressed=True,
    )
    ok(S, "LlmJudgementResult", f"verdict={jr.verdict}, conf={jr.confidence}, findings={len(jr.findings)}")
except ValidationError as e:
    fail(S, "LlmJudgementResult", str(e))

# FinalSynthesis
try:
    fs = FinalSynthesis(
        verdict=Verdict.PASS, confidence=0.87, reason="All good",
        primary_tag="PIPELINE_VALIDATED", findings_summary=[], data_completeness=1.0,
    )
    ok(S, "FinalSynthesis", f"verdict={fs.verdict}, conf={fs.confidence}")
except ValidationError as e:
    fail(S, "FinalSynthesis", str(e))

# AppUser
try:
    user = AppUser(
        google_subject_id="demo-sub-001", email="demo@gsk.com", display_name="Demo User"
    )
    ok(S, "AppUser", f"email={user.email}")
except ValidationError as e:
    fail(S, "AppUser", str(e))

# Confidence validator — should reject out-of-range
try:
    LlmJudgementResult(verdict=Verdict.PASS, confidence=1.5, reason="bad", findings=[], focus_area_addressed=False)
    warn(S, "Confidence range check", "Out-of-range value 1.5 was accepted — check validator")
except (ValidationError, ValueError):
    ok(S, "Confidence range check", "Out-of-range 1.5 correctly rejected")

print("\n✅ Models section complete.")